In [2]:
!pip install -q transformers langchain langchain-community sentence-transformers faiss-cpu

In [3]:
!pip install langchain
!pip install transformers
!pip install accelerate
!pip install sentence-transformers
!pip install chromadb
!pip install bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentel

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
patient_data = {
    "name": "Rahul",
    "age": 52,
    "procedure": "Appendix Surgery",
    "allergy": "Penicillin",
    "history": "Diabetes"
}

In [ ]:
prompt = f"""
You are a medical safety assistant.

Patient:
Name: {patient_data['name']}
Age: {patient_data['age']}
Procedure: {patient_data['procedure']}
Allergy: {patient_data['allergy']}
History: {patient_data['history']}

Generate:
1. Pre-procedure safety checklist
2. Possible risks
3. Critical warnings
"""

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(result)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



You are a medical safety assistant.

Patient:
Name: Rahul
Age: 52
Procedure: Appendix Surgery
Allergy: Penicillin
History: Diabetes

Generate:
1. Pre-procedure safety checklist
2. Possible risks
3. Critical warnings
4. Post-procedure care instructions

1. Pre-procedure safety checklist:

* Review patient's medical history and allergies
* Confirm the patient's name and age
* Verify the patient's consent for the procedure
* Check for any recent changes in the patient's health status
* Ensure that the patient has fasted for the appropriate amount of time
* Verify that the patient has been cleared by the anesthesia team
* Confirm the patient's location and room number
* Verify that all necessary equipment and supplies are available
* Ensure that the surgical team is aware of any special instructions or precautions

2. Possible risks:

* Complications related to anesthesia
* Infection
* Bleeding
* Damage to surrounding organs or tissues
* Reaction to medications or allergens
* Blood clots


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings()

vectordb = Chroma(
    collection_name="medical_guidelines",
    embedding_function=embedding
)

/tmp/ipykernel_5259/2863641776.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings()
/tmp/ipykernel_5259/2863641776.py:4: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding = HuggingFaceEmbeddings()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_5259/2863641776.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


In [ ]:
if patient_data["allergy"].lower() == "penicillin":
    print("ALERT: Penicillin allergy detected")

ALERT: Penicillin allergy detected


In [ ]:
import pandas as pd

log = pd.DataFrame([{
    "patient": patient_data["name"],
    "procedure": patient_data["procedure"],
    "ai_result": result
}])

log

,patient,procedure,ai_result
0,Rahul,Appendix Surgery,\nYou are a medical safety assistant.\n\nPatie...
